# Vectorización: Bag of Words

---

## Objetivo

Hasta ahora aprendiste a tokenizar, lematizar y filtrar texto con spaCy. Ese trabajo produce una *lista de palabras*. Ahora damos el paso siguiente: **convertir esa lista en números** que una computadora pueda procesar y comparar.

La técnica que vamos a ver se llama **Bag of Words** (Bolsa de Palabras, o BoW). Es el punto de partida de casi todos los sistemas de análisis de texto: filtros de spam, motores de búsqueda, clasificadores de sentimiento.

## Resultados de aprendizaje

Al finalizar este cuaderno vas a poder:

- Explicar con tus palabras qué es Bag of Words y qué información preserva (y cuál descarta).
- Construir manualmente una matriz de conteos a partir de un corpus pequeño.
- Automatizar ese proceso usando `CountVectorizer` de `scikit-learn`.
- Aplicar BoW a casos reales e interpretar los resultados.
- Identificar las limitaciones de BoW y argumentar por qué existe TF-IDF.

## Relación con cuadernos anteriores

- En los cuadernos `001` a `006` de spaCy aprendiste a tokenizar, lematizar y filtrar texto.
- En `006` formalizaste las decisiones de preprocesamiento (¿lematizar? ¿filtrar stopwords?).
- **Acá** tomás esas listas de tokens y las convertís en vectores numéricos: el puente entre el lenguaje y la matemática.

Se agregaron conclusiones propias de cada sección

---

## Sección 1: Concepto — La analogía de la bolsa

Imaginá que recibís un texto impreso y tenés que resumir "de qué trata" sin leerlo en orden. Un método posible:

1. **Tomás el texto** → "El perro muerde al perro".
2. **Cortás cada palabra** (tokens) → `["El", "perro", "muerde", "al", "perro"]`.
3. **Las metés en una bolsa** y la agitás: el orden original se pierde.
4. **Contás cuántas hay de cada una** → obtenés un perfil numérico del texto.

### El resultado: un inventario de frecuencias

| Palabra | Cantidad |
|---------|----------|
| el      | 1        |
| perro   | 2        |
| muerde  | 1        |
| al      | 1        |

**Eso es Bag of Words.** Cada texto queda representado por una lista de números: cuántas veces aparece cada palabra del vocabulario compartido.

> **Para pensar antes de continuar**: ¿Qué información pierde esta representación? ¿"No me gusta nada" y "Me gusta, no nada" producirían el mismo vector?

Lo unico que pierde es el orden y el sentido, como son las mismas palabras creo que genera el mismo vector.

---

## Sección 2: Ejercicio manual

Antes de escribir código, vamos a hacer este proceso a mano para fijar la intuición.

### Corpus: tres titulares de noticias argentinas

Trabajamos con tres titulares breves de dos temas distintos:

1. `"Argentina ganó el Mundial"`
2. `"Messi marcó dos goles"`
3. `"La inflación subió en Argentina"`

### Paso 1 — Crear el vocabulario

El vocabulario reúne **todas las palabras únicas** del corpus (en minúscula y orden alfabético):

**Vocabulario:** argentina, dos, el, en, ganó, goles, inflación, la, marcó, messi, mundial, subió *(12 términos)*

### Paso 2 — Construir la matriz de conteos

Cada fila es un documento; cada columna, una palabra del vocabulario.

| Documento | argentina | dos | el | en | ganó | goles | inflación | la | marcó | messi | mundial | subió |
|-----------|-----------|-----|----|----|------|-------|-----------|----|-------|-------|---------|-------|
| Titular 1 | 1         | 0   | 1  | 0  | 1    | 0     | 0         | 0  | 0     | 0     | 1       | 0     |
| Titular 2 | 0         | 1   | 0  | 0  | 0    | 1     | 0         | 0  | 1     | 1     | 0       | 0     |
| Titular 3 | 1         | 0   | 0  | 1  | 0    | 0     | 1         | 1  | 0     | 0     | 0       | 1     |

`CountVectorizer` de Python hace exactamente estos dos pasos: arma el vocabulario y construye la tabla de conteos.

> **Para pensar**: ¿Qué tienen en común el Titular 1 y el Titular 3? ¿Qué los diferencia del Titular 2? Fijate en las columnas con valor `1`.

Esta matriz se activa en 1 cuando si aparece la palabra en el titular. Por lo tanto,  en la primer columna Argentina aparece en el Titular 1 y 3 pero no en el Titular 2. Sin embargo, la palabra argentina me ayuda un poco a distinguir temas pero el contexto es distinto.

---

## Sección 3: Automatización con Python

Ahora que entendés el proceso manual, vamos a automatizarlo. En proyectos reales el vocabulario tiene miles de palabras y el corpus miles de documentos, así que hacerlo a mano sería imposible.

Vamos a usar `CountVectorizer` de `scikit-learn`. Hace exactamente los dos pasos que hicimos a mano:

- **`fit`**: recorre todos los documentos y arma el vocabulario (las columnas de la tabla).
- **`transform`**: cuenta cuántas veces aparece cada palabra del vocabulario en cada documento.

Cuando usamos `fit_transform`, ejecutamos ambos pasos en una sola llamada.

In [1]:
# Importaciones

# CountVectorizer: construye el vocabulario y cuenta frecuencias automáticamente.
from sklearn.feature_extraction.text import CountVectorizer
import pandas as pd

# Corpus de trabajo 
# Los mismos tres titulares del ejercicio manual, para poder comparar.

corpus_noticias = [
    "Argentina ganó el Mundial",
    "Messi marcó dos goles",
    "La inflación subió en Argentina",
]

print("Corpus de trabajo:")
for i, titular in enumerate(corpus_noticias, start=1):
    print(f"  Titular {i}: {titular}")

Corpus de trabajo:
  Titular 1: Argentina ganó el Mundial
  Titular 2: Messi marcó dos goles
  Titular 3: La inflación subió en Argentina


Le damos nuestro corpus a `CountVectorizer`. Esperamos obtener:

- Un **vocabulario** con los 12 términos únicos (en minúsculas y orden alfabético).
- Una **matriz esparsa** de 3 filas × 12 columnas.

*Nota*: "matriz esparsa" es un formato de almacenamiento eficiente que solo guarda los valores distintos de cero. Con `.toarray()` la convertimos a una tabla completa.

In [2]:
# Creamos el contador automático de palabras

# Inicializamos sin parámetros: usa la configuración por defecto
# (tokeniza por espacios, pasa todo a minúsculas, ignora puntuación).
contador_frecuencias = CountVectorizer()

# fit_transform hace los dos pasos del ejercicio manual en una sola instrucción:
#   1. Aprende el vocabulario (fit)
#   2. Construye la matriz de conteos (transform)
matriz_conteo = contador_frecuencias.fit_transform(corpus_noticias)

# Recuperamos el vocabulario identificado
vocabulario = contador_frecuencias.get_feature_names_out()

print("Vocabulario identificado (orden alfabético):")
print(list(vocabulario))
print(f"\nTotal de términos únicos: {len(vocabulario)}")

Vocabulario identificado (orden alfabético):
['argentina', 'dos', 'el', 'en', 'ganó', 'goles', 'inflación', 'la', 'marcó', 'messi', 'mundial', 'subió']

Total de términos únicos: 12


Deberías ver exactamente los mismos 12 términos que armamos a mano. `CountVectorizer` los ordena alfabéticamente y los pasa a minúsculas de forma automática. Ahora construimos la tabla completa:

In [3]:
#Visualizamos la matriz como tabla legible

# .toarray() convierte la matriz esparsa en un arreglo numérico completo.
tabla_bow = pd.DataFrame(
    matriz_conteo.toarray(),
    columns=vocabulario,                           # palabras del vocabulario → encabezados
    index=["Titular 1", "Titular 2", "Titular 3"]  # etiqueta de cada fila
)

print("Tabla de frecuencias (Bag of Words):")
tabla_bow

Tabla de frecuencias (Bag of Words):


,argentina,dos,el,en,ganó,goles,inflación,la,marcó,messi,mundial,subió
Titular 1,1,0,1,0,1,0,0,0,0,0,1,0
Titular 2,0,1,0,0,0,1,0,0,1,1,0,0
Titular 3,1,0,0,1,0,0,1,1,0,0,0,1


---

## Sección 4: Reflexión guiada

Observá la tabla con atención. Cada fila es ahora una *huella digital* numérica de cada titular.

1. **¿Qué titulares son más parecidos entre sí?** Compará las filas: ¿cuáles comparten más columnas con valores distintos de cero?
2. **La palabra "argentina" aparece en los Titulares 1 y 3.** ¿Eso significa que tratan el mismo tema? ¿Podría confundir a un algoritmo?
3. **¿Cuáles son los términos que mejor distinguen cada texto?** Identificá las palabras exclusivas de cada titular.
4. **¿Hay alguna columna con el mismo valor en las tres filas?** Si la hubiera, ¿ayudaría a clasificar los textos?

### Conclusiones clave

- El Titular 1 y el Titular 3 comparten "argentina": el algoritmo los percibe como parecidos, aunque uno sea deportivo y el otro económico.
- Términos como "messi", "goles" o "inflación" son los que mejor distinguen los textos: son *exclusivos* de un solo documento.
- Las palabras que aparecen en muchos documentos **reducen la capacidad discriminativa** de la representación.

> Esa es exactamente la limitación que resuelve TF-IDF: pondera la importancia de cada palabra según qué tan *rara* es en el corpus.

1. Los titulares mas parecidos son el titular 1 y 3 porque ambos comparten la palabra argentina. El titular 2 no comparte ninguna palabra por eso es el unico distinto

2. El hecho de que aparezca la palabra en ambos titulares no significa que se trate del mismo tema, es mas uno es un titular deportivo y el otro refiere a lo economico. Por lo tanto, si confudiria al algoritmo, porque solo percibe los documentos basandose en sus frecuencias compartidas  sin entender el contexto

3. Los terminos que reflejan mejor el contexto  son los exclusivos de cada documento. es decir  su vector seria (0 1 0), para dar un ejemplo, es la palabra "goles" exclusivo al titular 2.

4. En este ejecicio, no hay  ninguna columna con el mismo valor.Es decir, no hay una palabra que aparezca en los 3 textos simultaneamente. Pero si existiera esa columna  no ayudaria a clasificar los textos.

---

## Sección 5: Aplicación práctica — Detección de mensajes no deseados

¿Cómo podría un sistema detectar automáticamente si un mensaje de WhatsApp es spam?

La lógica es simple:
- Vectorizamos los mensajes con BoW.
- Observamos qué palabras son frecuentes en mensajes no deseados ("GANASTE", "GRATIS", "OFERTA").
- Observamos qué palabras son frecuentes en mensajes legítimos ("apuntes", "recuperatorio", "parcial").

Vamos a probar esta idea con un corpus del entorno universitario.

In [4]:
#Corpus de mensajes de WhatsApp universitarios

mensajes = [
    "Hola profe, ¿la entrega del TP es hasta el viernes o el lunes?",       # Legítimo
    "GANASTE UN CELULAR GRATIS!! Entrá al link y reclamá tu premio ahora!!!", # Spam
    "¿Quedaste para el recuperatorio del martes? Te mando los apuntes",      # Legítimo
    "OFERTA LIMITADA!! 90% de descuento solo HOY!! No pierdas la oportunidad!!", # Spam
    "El parcial estuvo difícil, ¿vos cómo lo sentiste?",                    # Legítimo
]

etiquetas = ["Legítimo", "Spam", "Legítimo", "Spam", "Legítimo"]

print("Mensajes de prueba:")
for i, (mensaje, etiqueta) in enumerate(zip(mensajes, etiquetas), start=1):
    print(f"  {i}. [{etiqueta}] {mensaje}")

Mensajes de prueba:
  1. [Legítimo] Hola profe, ¿la entrega del TP es hasta el viernes o el lunes?
  2. [Spam] GANASTE UN CELULAR GRATIS!! Entrá al link y reclamá tu premio ahora!!!
  3. [Legítimo] ¿Quedaste para el recuperatorio del martes? Te mando los apuntes
  4. [Spam] OFERTA LIMITADA!! 90% de descuento solo HOY!! No pierdas la oportunidad!!
  5. [Legítimo] El parcial estuvo difícil, ¿vos cómo lo sentiste?


Ahora vectorizamos estos mensajes. La tabla completa tendría muchas columnas (una por palabra única), así que seleccionamos solo los términos que mejor distinguen los dos tipos de mensajes.

In [6]:
# Vectorizamos los mensajes

# Creamos un nuevo contador (independiente del de noticias)
contador_mensajes = CountVectorizer()
matriz_mensajes   = contador_mensajes.fit_transform(mensajes)
vocabulario_mensajes = contador_mensajes.get_feature_names_out()

# Armamos la tabla completa
tabla_mensajes = pd.DataFrame(
    matriz_mensajes.toarray(),
    columns=vocabulario_mensajes,
    index=[f"Mensaje {i+1} ({etiquetas[i]})" for i in range(len(mensajes))]
)

# Seleccionamos términos indicadores para cada categoría
terminos_spam     = ["ganaste", "gratis", "oferta", "limitada", "descuento", "premio"]
terminos_legitimo = ["apuntes", "recuperatorio", "parcial", "entrega", "profe"]

# Filtramos solo los que existen en el vocabulario aprendido
columnas_visibles = [
    t for t in terminos_spam + terminos_legitimo
    if t in tabla_mensajes.columns
]

print(f"Vocabulario total: {len(vocabulario_mensajes)} términos únicos.")
print(f"Mostramos {len(columnas_visibles)} términos clave:\n")
tabla_mensajes[columnas_visibles]

Vocabulario total: 47 términos únicos.
Mostramos 11 términos clave:



,ganaste,gratis,oferta,limitada,descuento,premio,apuntes,recuperatorio,parcial,entrega,profe
Mensaje 1 (Legítimo),0,0,0,0,0,0,0,0,0,1,1
Mensaje 2 (Spam),1,1,0,0,0,1,0,0,0,0,0
Mensaje 3 (Legítimo),0,0,0,0,0,0,1,1,0,0,0
Mensaje 4 (Spam),0,0,1,1,1,0,0,0,0,0,0
Mensaje 5 (Legítimo),0,0,0,0,0,0,0,0,1,0,0


### ¿Qué observamos?

- Los **mensajes de spam** (2 y 4) concentran sus valores en los términos de urgencia y oferta: "ganaste", "gratis", "oferta", "limitada".
- Los **mensajes legítimos** (1, 3 y 5) concentran sus valores en vocabulario académico-universitario: "recuperatorio", "apuntes", "parcial".
- **No hay solapamiento**: cada grupo usa un vocabulario diferente.

Un clasificador automático aprende exactamente estos patrones: la presencia o ausencia de ciertas palabras es una señal fuerte para decidir si un mensaje es spam o no.

Para que esto sea efectivo  los mensajes de spam y los legitimos deberian usar conjuntos de palabras casi totalmente distintos. En este caso los mensajes de spam se caracterizan por tener estos terminos de urgencia o premios, en contraste con los terminos legitimos especificos al entorno academico. No hay solapamiento asi que el algoritnmo podria distinguir las categorias de forma sencilla. 
Este seria un caso ideal porque qué pasaria si el repartidor de mensajes spam  empieza a poner palabras "legitimas" como camuflaje. Eso enganiaria al algoritmo y de nuevo se necesita recurrir al contexto.

---

## Sección 6: Limitaciones y próximos pasos

Bag of Words es potente y simple, pero tiene puntos ciegos importantes:

1. **Se pierde el orden**: "No me gusta nada" y "Me gusta, no nada" producen el mismo vector aunque signifiquen lo opuesto.
2. **El contexto es invisible**: "banco" (de sentarse) y "banco" (financiero) se cuentan como la misma palabra.
3. **Todas las palabras pesan igual**: "argentina" y "messi" reciben el mismo valor aunque "argentina" aparezca en muchos más documentos.
4. **Vocabularios gigantes**: con miles de documentos, la matriz crece enormemente y se vuelve costosa de procesar.

En el próximo cuaderno vamos a ver **TF-IDF**, una evolución de BoW que resuelve el punto 3: asigna pesos distintos según qué tan *relevante* es cada palabra dentro del corpus.

---

## Actividad de cierre — Análisis de sentimiento en reseñas de cine argentino

Para cerrar, aplicamos BoW a un problema diferente: **clasificar reseñas de películas argentinas como positivas o negativas**.

### Para pensar antes de ejecutar

Antes de correr el código, reflexioná:

1. ¿Qué palabras esperás encontrar en las reseñas positivas?
2. ¿Y en las negativas?
3. ¿Creés que BoW va a poder distinguirlas solo con conteos de palabras?

Anotá tus hipótesis y comparalas con la tabla que obtengas después.

Hipotesis

En las resenias positivas seguro aparecen palabras como "recomiendo"," emocionante", "buenisima".En las negativas puede aparecer " mala", "malisima", "aburrida","desastre".No creo que sea  suficiente para hacer una buena clasificacion.

In [ ]:
#Corpus de reseñas de cine argentino

resenias = [
    "Excelente película, me atrapó de principio a fin, la recomiendo",  # Positiva
    "Una pérdida de tiempo total, aburrida y sin sentido, no la vean",  # Negativa
    "Me sorprendió muchísimo, no esperaba que fuera tan buena",         # Positiva
    "Malísima, los actores no convencen y el guión es un desastre",     # Negativa
    "Una obra maestra del cine argentino, emocionante y profunda",      # Positiva
]

polaridad = ["Positiva", "Negativa", "Positiva", "Negativa", "Positiva"]

print("Reseñas a analizar:")
for i, (resenia, pol) in enumerate(zip(resenias, polaridad), start=1):
    print(f"  {i}. [{pol}] {resenia}")

Reseñas a analizar:
  1. [Positiva] Excelente película, me atrapó de principio a fin, la recomiendo
  2. [Negativa] Una pérdida de tiempo total, aburrida y sin sentido, no la vean
  3. [Positiva] Me sorprendió muchísimo, no esperaba que fuera tan buena
  4. [Negativa] Malísima, los actores no convencen y el guión es un desastre
  5. [Positiva] Una obra maestra del cine argentino, emocionante y profunda


In [9]:
#Vectorizamos las reseñas y mostramos los términos más reveladores

contador_resenias    = CountVectorizer()
matriz_resenias      = contador_resenias.fit_transform(resenias)
vocabulario_resenias = contador_resenias.get_feature_names_out()

tabla_resenias = pd.DataFrame(
    matriz_resenias.toarray(),
    columns=vocabulario_resenias,
    index=[f"Reseña {i+1} ({polaridad[i]})" for i in range(len(resenias))]
)

# Términos indicadores de cada polaridad
terminos_positivos = ["excelente", "recomiendo", "buena", "sorprendió", "maestra", "emocionante"]
terminos_negativos = ["aburrida", "pérdida", "malísima", "desastre", "vean"]

columnas_sentimiento = [
    t for t in terminos_positivos + terminos_negativos
    if t in tabla_resenias.columns
]

print(f"Vocabulario total: {len(vocabulario_resenias)} términos.")
print(f"Mostramos {len(columnas_sentimiento)} términos clave:\n")
tabla_resenias[columnas_sentimiento]

Vocabulario total: 41 términos.
Mostramos 11 términos clave:



,excelente,recomiendo,buena,sorprendió,maestra,emocionante,aburrida,pérdida,malísima,desastre,vean
Reseña 1 (Positiva),1,1,0,0,0,0,0,0,0,0,0
Reseña 2 (Negativa),0,0,0,0,0,0,1,1,0,0,1
Reseña 3 (Positiva),0,0,1,1,0,0,0,0,0,0,0
Reseña 4 (Negativa),0,0,0,0,0,0,0,0,1,1,0
Reseña 5 (Positiva),0,0,0,0,1,1,0,0,0,0,0


### ¿Coincidió con tus hipótesis?

- Las **reseñas positivas** (1, 3 y 5) concentran sus valores en la izquierda: "excelente", "recomiendo", "maestra", "emocionante".
- Las **reseñas negativas** (2 y 4) concentran sus valores en la derecha: "aburrida", "malísima", "desastre".

Este patrón es exactamente el que aprende un clasificador automático: la presencia de ciertas palabras es un indicador fuerte de la polaridad del texto.

> **Para reflexionar**: ¿Qué pasaría con una reseña como "La película no es mala, pero tampoco es buena"? ¿Creés que BoW la clasificaría correctamente? ¿Por qué?

No reafirmo mi hipotesis porque  yo generalize, es decir, una opinion a veces  empiezan pareciendo positivas luego puede ser que nombren las cosas negativas que encontraron a la hora de visualizar la pelicula. Pero bueno, en principio depende mucho del corpus en este caso  cada  opinion es o negativa o positiva y no hay ambguedades. Se rompe si hay opiniones neutras.

---

## Cierre

En este cuaderno aprendiste cómo Bag of Words convierte texto en números que un algoritmo puede procesar. Lo aplicaste a dos casos reales:

- **Detección de mensajes no deseados** en conversaciones universitarias.
- **Análisis de sentimiento** en reseñas de cine argentino.

En ambos casos, la simple frecuencia de palabras fue suficiente para distinguir categorías. Pero también viste las limitaciones: BoW ignora el orden, el contexto y la importancia relativa de las palabras.

**En el próximo cuaderno** vamos a ver **TF-IDF**, una evolución de BoW que no solo cuenta palabras sino que pondera su importancia dentro del corpus: cuánto más rara es una palabra en el conjunto de documentos, más relevante resulta para el texto donde aparece.